# Sales forcasting

In [1]:
#Import Libraries 
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV 
from sklearn.preprocessing import StandardScaler, LabelEncoder 
from sklearn.linear_model import LinearRegression 
from sklearn.tree import DecisionTreeRegressor 
from sklearn.ensemble import RandomForestRegressor 
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score 
# Optional (if installed) 
from xgboost import XGBRegressor 

In [2]:
#Load Dataset 
df = pd.read_csv("sales_data.csv") 

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe3 in position 14: invalid continuation byte

In [ ]:
#Data Understanding & Preprocessing 
df.head() 
df.shape 
df.info() 
df.isnull().sum()

In [ ]:
#Convert Date Column 
df['Date'] = pd.to_datetime(df['Date']) 

In [ ]:
#Extract Time Features 
df['Year'] = df['Date'].dt.year 
df['Month'] = df['Date'].dt.month 
df['Day'] = df['Date'].dt.day 
df['DayOfWeek'] = df['Date'].dt.dayofweek

In [ ]:
#Handle Missing Values 
df.fillna(method='ffill', inplace=True) 

In [ ]:
#Data Visualization 
#Sales Trend 
plt.figure() 
plt.plot(df['Date'], df['Sales']) 
plt.title("Sales Trend Over Time") 
plt.show() 

In [ ]:
#Monthly Sales 
monthly_sales = df.groupby('Month')['Sales'].sum() 
plt.figure() 
monthly_sales.plot(kind='bar') 
plt.title("Monthly Sales Comparison") 
plt.show()

In [ ]:
#Promotion Impact 
sns.boxplot(x='Promotion', y='Sales', data=df) 
plt.title("Promotion vs Sales") 
plt.show() 

In [ ]:
#Correlation Heatmap 
plt.figure() 
sns.heatmap(df.corr(numeric_only=True), annot=True) 
plt.title("Correlation Heatmap") 
plt.show()

# Feature Engineering 

In [ ]:
#Lag Features 
df['Lag_1'] = df['Sales'].shift(1) 
df['Lag_7'] = df['Sales'].shift(7) 

In [ ]:
#Rolling Mean 
df['Rolling_7'] = df['Sales'].rolling(7).mean() 
df['Rolling_30'] = df['Sales'].rolling(30).mean() 


In [ ]:
#Drop NA (after lagging) 
df.dropna(inplace=True) 

In [ ]:
#Encoding (if categorical present) 
le = LabelEncoder() 
if 'Store' in df.columns: 
    df['Store'] = le.fit_transform(df['Store']) 
if 'Product' in df.columns: 
    df['Product'] = le.fit_transform(df['Product']) 

In [ ]:
#Feature & Target 
X = df.drop(['Sales', 'Date'], axis=1) 
y = df['Sales']

In [ ]:
#Scaling (optional) 
scaler = StandardScaler() 
X_scaled = scaler.fit_transform(X)

In [ ]:
#4. Train-Test Split (Time-Based) 
split = int(len(df) * 0.8) 
X_train, X_test = X_scaled[:split], X_scaled[split:] 
y_train, y_test = y[:split], y[split:]

# Model Building 

In [ ]:
#Train Models 
lr = LinearRegression() 
dt = DecisionTreeRegressor() 
rf = RandomForestRegressor() 
xgb = XGBRegressor() 
models = { 
    "Linear": lr, 
    "Decision Tree": dt, 
    "Random Forest": rf, 
    "XGBoost": xgb 
} 
 
predictions = {} 
 
for name, model in models.items(): 
    model.fit(X_train, y_train) 
    predictions[name] = model.predict(X_test) 

# Model Evaluation 

In [ ]:
#Evaluation Function 
def evaluate(y_true, y_pred): 
    mae = mean_absolute_error(y_true, y_pred) 
    mse = mean_squared_error(y_true, y_pred) 
    rmse = np.sqrt(mse) 
    r2 = r2_score(y_true, y_pred) 
    return mae, mse, rmse, r2 

In [ ]:
#Compare Models 
results = [] 
 
for name in models: 
    mae, mse, rmse, r2 = evaluate(y_test, predictions[name]) 
    results.append([name, mae, mse, rmse, r2]) 
 
results_df = pd.DataFrame(results, columns=["Model", "MAE", "MSE", "RMSE", 
"R2"]) 
print(results_df)

In [ ]:
#Visualization 
#Actual vs Predicted 
plt.figure() 
plt.plot(y_test.values, label='Actual') 
plt.plot(predictions['Random Forest'], label='Predicted') 
plt.legend() 
plt.title("Actual vs Predicted Sales") 
plt.show()

In [ ]:
# Residual Plot 
residuals = y_test.values - predictions['Random Forest'] 
 
plt.figure() 
sns.histplot(residuals, kde=True) 
plt.title("Residual Distribution") 
plt.show() 

In [ ]:
#Cross Validation (Time Series) 
tscv = TimeSeriesSplit(n_splits=5) 
 
for train_idx, test_idx in tscv.split(X_scaled): 
    X_tr, X_te = X_scaled[train_idx], X_scaled[test_idx] 
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx] 
 
    rf.fit(X_tr, y_tr) 
    preds = rf.predict(X_te)

In [ ]:
# Hyperparameter Tuning 
param_grid = { 
    'n_estimators': [50, 100], 
    'max_depth': [10, 20] 
} 
 
grid = GridSearchCV(RandomForestRegressor(), param_grid, cv=3) 
grid.fit(X_train, y_train) 
 
best_model = grid.best_estimator_

In [ ]:
 #Future Forecast (Next 30 Days) 
future_preds = [] 
 
last_data = X_scaled[-30:] 
 
for i in range(30): 
    pred = best_model.predict([last_data[-1]]) 
    future_preds.append(pred[0])

In [ ]:
#Forecast Plot 
plt.figure() 
plt.plot(range(30), future_preds) 
plt.title("Next 30 Days Sales Forecast") 
plt.show()

In [ ]:
# Prediction Function 
def predict_sales(store, product, promo, date): 
    date = pd.to_datetime(date) 
 
    year = date.year 
    month = date.month 
    day = date.day 
    dow = date.dayofweek 
 
    input_data = np.array([[store, product, promo, year, month, day, dow, 0, 0, 0, 
0]]) 
    input_scaled = scaler.transform(input_data) 
 
    return best_model.predict(input_scaled)[0]